# Paper 1 — Figures 1, 2, 6, 8 (maps, flowchart, distance scatter)

**Script file:** `Paper1_Figures_1_2_6_8.ipynb`

Reproduces, from source data, the four figures that were originally built outside the
notebook series:

- **Figure 1** — Study area location and sampling design (Thailand inset + zoomed study
  area with administrative boundaries, sites, cement plants, Mittraphap Road).
- **Figure 2** — Analytical workflow flowchart (Input → Process → Output).
- **Figure 6** — Land-use context map (sites colored by PM2.5, cement plants, road).
- **Figure 8** — PM2.5 vs. distance to nearest verified cement plant (scatter + OLS fit).

All figures are exported as both **SVG** (true vector, for the manuscript) and PNG
(fallback / quick preview).

**สิ่งที่ต้องเตรียมก่อนรัน:** ไฟล์ `บันทึกการตรวจวัดปริมาณฝุ่น.xlsx` (ข้อมูลภาคสนามต้นฉบับ 32 จุด)
และ `site_coordinates_resolved.csv` (พิกัดที่ resolve แล้ว) ส่วนขอบเขตการปกครองจะดึงสดจาก GitHub (ต้องมีอินเทอร์เน็ตใน Colab)


In [ ]:
# SCRIPT: Paper1_Figures_1_2_6_8.ipynb
# SECTION: 0 - Setup
import sys, subprocess

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg, _imp in [("geopandas", "geopandas"), ("openpyxl", "openpyxl")]:
    _ensure(_pkg, _imp)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.colors import LogNorm
from shapely.geometry import LineString
from scipy.stats import pearsonr


## 1. โหลดข้อมูลภาคสนามและพิกัด


In [ ]:
# SCRIPT: Paper1_Figures_1_2_6_8.ipynb
# SECTION: 1 - Load field data and coordinates
from google.colab import files
import os

FIELD_XLSX = "บันทึกการตรวจวัดปริมาณฝุ่น.xlsx"
if not os.path.exists(FIELD_XLSX):
    print(f"Upload {FIELD_XLSX}:")
    uploaded = files.upload()
    FIELD_XLSX = list(uploaded.keys())[0]

COORDS_CSV = "site_coordinates_resolved.csv"
if not os.path.exists(COORDS_CSV):
    print(f"Upload {COORDS_CSV}:")
    uploaded = files.upload()
    COORDS_CSV = list(uploaded.keys())[0]

raw = pd.read_excel(FIELD_XLSX, sheet_name="Sheet1", header=0)
raw.columns = [str(c).strip() for c in raw.columns]
raw = raw.rename(columns={raw.columns[0]: "site_id"})
raw = raw[raw["site_id"].notna()].reset_index(drop=True)

coords = pd.read_csv(COORDS_CSV)
df = raw.merge(coords, on="site_id", how="left")
df = df.rename(columns={"PM2.5 (µg/m3)": "pm25"})
df["is_survey"] = df["site_id"].str.contains("survey")
print(f"n={len(df)}, missing coordinates: {df['lat'].isna().sum()}")
df[["site_id", "lat", "lon", "pm25", "is_survey"]].head()


## 2. ขอบเขตการปกครองและระยะทาง (จาก GitHub — ต้องมีอินเทอร์เน็ต)


In [ ]:
# SCRIPT: Paper1_Figures_1_2_6_8.ipynb
# SECTION: 2 - Fetch administrative boundaries + define cement plants and road
import urllib.request

# Thailand country + province boundaries (GADM-derived, public dataset)
PROV_URL = "https://raw.githubusercontent.com/cvibhagool/thailand-map/master/thailand-provinces.geojson"
if not os.path.exists("thailand_provinces.geojson"):
    urllib.request.urlretrieve(PROV_URL, "thailand_provinces.geojson")
provinces = gpd.read_file("thailand_provinces.geojson")
thailand_country = provinces.dissolve(by="NAME_0")

# District (amphoe) level boundaries with province names attached
AMPHOE_URL = "https://raw.githubusercontent.com/prasertcbs/thailand_gis/master/amphoe/thailand_province_amphoe_simplify.json"
if not os.path.exists("thailand_amphoe.json"):
    urllib.request.urlretrieve(AMPHOE_URL, "thailand_amphoe.json")
amphoe = gpd.read_file("thailand_amphoe.json")

lat_min, lat_max = df["lat"].min() - 0.15, df["lat"].max() + 0.15
lon_min, lon_max = df["lon"].min() - 0.15, df["lon"].max() + 0.15
study_amphoe = amphoe.cx[lon_min:lon_max, lat_min:lat_max]
study_province = study_amphoe.dissolve(by="ADM1_EN")

# Three verified cement plants (coordinates from Global Cement and Concrete Tracker /
# Global Energy Monitor -- see manuscript Section 2.3.3 for sourcing)
plants = pd.DataFrame({
    "name": ["TPI Polene Mittaphap", "Siam Cement (Kaeng Khoi) Ban Pa", "Siam Cement Khao Wong"],
    "lat": [14.64631, 14.6152, 14.67494],
    "lon": [101.12650, 101.0181, 100.85217],
})

# Mittraphap Road (Highway 2) through the study corridor -- interpolated waypoint
# approximation through named towns (see manuscript Section 5, Limitations, for the
# note that this is not a surveyed centerline).
mittraphap_waypoints = [
    (100.755, 14.353), (100.830, 14.430), (100.910, 14.529), (101.007, 14.596),
    (101.165, 14.660), (101.418, 14.705), (101.627, 14.745),
]
road_gdf = gpd.GeoDataFrame({"name": ["Mittraphap Road (Highway 2)"]},
                             geometry=[LineString(mittraphap_waypoints)], crs="EPSG:4326")

print("Provinces in study area:", study_amphoe["ADM1_EN"].unique())


## 3. Figure 1 — Study area location and sampling design


In [ ]:
# SCRIPT: Paper1_Figures_1_2_6_8.ipynb
# SECTION: 3 - Figure 1
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6.2), gridspec_kw={"width_ratios": [1, 1.6]})

thailand_country.plot(ax=ax1, color="#e8e8e8", edgecolor="#888888", linewidth=0.5)
rect = mpatches.Rectangle((lon_min, lat_min), lon_max - lon_min, lat_max - lat_min,
                           fill=False, edgecolor="red", linewidth=1.8, zorder=5)
ax1.add_patch(rect)
ax1.set_xlim(97.0, 106.0); ax1.set_ylim(5.5, 20.7)
ax1.set_title("(a) Thailand", fontsize=11)
ax1.set_xlabel("Longitude (\u00b0E)"); ax1.set_ylabel("Latitude (\u00b0N)")
ax1.set_aspect("equal")

study_amphoe.plot(ax=ax2, color="#f7f7f7", edgecolor="#bbbbbb", linewidth=0.4)
study_province.boundary.plot(ax=ax2, edgecolor="#555555", linewidth=1.1)
road_gdf.plot(ax=ax2, color="#d95f02", linewidth=2.2, zorder=4, label="Mittraphap Road (Hwy 2)")

grid_sites = df[~df["is_survey"]]
survey_sites = df[df["is_survey"]]
ax2.scatter(grid_sites["lon"], grid_sites["lat"], s=45, c="#1f78b4", edgecolor="white",
            linewidth=0.6, zorder=5, label="Systematic-grid site (n=25)")
ax2.scatter(survey_sites["lon"], survey_sites["lat"], s=70, c="#e31a1c", marker="^",
            edgecolor="white", linewidth=0.6, zorder=6, label="Purposive risk site (n=7)")
ax2.scatter(plants["lon"], plants["lat"], s=140, c="black", marker="s", zorder=7,
            edgecolor="white", linewidth=0.8, label="Verified cement plant")

label_offsets = {"TPI Polene Mittaphap": (6, 10),
                  "Siam Cement (Kaeng Khoi) Ban Pa": (6, -14),
                  "Siam Cement Khao Wong": (-95, 8)}
for _, r in plants.iterrows():
    dx, dy = label_offsets[r["name"]]
    ax2.annotate(r["name"], (r["lon"], r["lat"]), fontsize=6.8, xytext=(dx, dy),
                 textcoords="offset points", fontweight="bold")
ax2.annotate("Saraburi", (100.98, 14.75), fontsize=9.5, color="#333333", style="italic")

ax2.set_xlim(lon_min, lon_max); ax2.set_ylim(lat_min, lat_max)
ax2.set_title("(b) Study area: sampling sites, district boundaries, and Mittraphap Road corridor", fontsize=10.5)
ax2.set_xlabel("Longitude (\u00b0E)"); ax2.set_ylabel("Latitude (\u00b0N)")
ax2.set_aspect("equal")
ax2.legend(loc="lower left", fontsize=7.5, framealpha=0.9)

plt.tight_layout()
plt.savefig("figure1_study_area.svg", format="svg", bbox_inches="tight")
plt.savefig("figure1_study_area.png", format="png", dpi=220, bbox_inches="tight")
plt.show()
print("Saved figure1_study_area.svg/.png")


## 4. Figure 2 — Analytical workflow flowchart

ไม่ต้องใช้ข้อมูลภาคสนาม — เป็นแค่ไดอแกรมสรุปขั้นตอนการวิเคราะห์


In [ ]:
# SCRIPT: Paper1_Figures_1_2_6_8.ipynb
# SECTION: 4 - Figure 2 (flowchart)
fig, ax = plt.subplots(figsize=(9, 11))
ax.set_xlim(0, 10); ax.set_ylim(0, 26); ax.axis("off")

def box(x, y, w, h, text, color="#dbe9f7", fontsize=8.3, edge="#2c5f8a"):
    b = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08,rounding_size=0.12",
                        linewidth=1.1, edgecolor=edge, facecolor=color, zorder=2)
    ax.add_patch(b)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize, zorder=3, wrap=True)

def arrow(p1, p2):
    a = FancyArrowPatch(p1, p2, arrowstyle="-|>", mutation_scale=14,
                         linewidth=1.1, color="#444444", zorder=1)
    ax.add_patch(a)

def stage_label(x, y, text):
    ax.text(x, y, text, ha="center", va="center", fontsize=10.5, fontweight="bold", color="#8a2c2c")

stage_label(9.3, 25.2, "INPUT")
box(0.4, 23.6, 4.0, 1.4, "Study design: 25 systematic-grid sites (7 provinces) +\n7 purposive risk sites (Saraburi corridor)")
box(4.8, 23.6, 4.8, 1.4, "Instrument: Particles Plus\u00ae 8306 optical particle\ncounter (PM0.3\u2013PM10, TPM, RH, temp, wind dir.)")
arrow((2.4, 23.6), (4.8, 23.0)); arrow((7.2, 23.6), (4.8, 23.0))
box(2.8, 21.6, 4.0, 1.1, "32 single-timepoint field readings,\nJune 2023 (early rainy season)")
arrow((4.8, 22.5), (4.8, 22.7))

stage_label(9.3, 20.6, "PROCESS")
box(2.8, 19.6, 4.0, 1.1, "Data preparation & quality screening\n(2.3.1)", color="#fde6cf", edge="#a8611a")
arrow((4.8, 21.6), (4.8, 20.7))
box(0.3, 17.9, 4.0, 1.4, "Geolocation: shortened map links \u2192\nresolved exact GPS coordinates", color="#fde6cf", edge="#a8611a")
box(5.0, 17.9, 4.5, 1.4, "RH screening: correlation of relative\nhumidity vs. PM2.5 (no correction applied\u2014\nno significant RH-dependence found)", color="#fde6cf", edge="#a8611a")
arrow((4.8, 19.6), (2.3, 19.3)); arrow((4.8, 19.6), (7.25, 19.3))

arrow((2.3, 17.9), (4.8, 17.0)); arrow((7.25, 17.9), (4.8, 17.0))
box(2.8, 15.9, 4.0, 1.1, "Spatial autocorrelation analysis\n(2.3.2)", color="#fde6cf", edge="#a8611a")

box(0.3, 13.6, 4.0, 1.7, "Global Moran's I (Eq. 1)\nraw & log-PM2.5; k-NN (k=5) and\nfixed distance-band (55 km) weights;\n999 permutations", color="#d9f0d9", edge="#2c6e2c")
box(5.0, 13.6, 4.5, 1.7, "Local Getis-Ord Gi* (Eq. 2)\nlog-PM2.5; k-NN (k=5) weights;\nz>1.96 & p<0.05 for hotspots;\n999 permutations", color="#d9f0d9", edge="#2c6e2c")
arrow((4.8, 15.9), (2.3, 15.3)); arrow((4.8, 15.9), (7.25, 15.3))

arrow((2.3, 13.6), (4.8, 12.7)); arrow((7.25, 13.6), (4.8, 12.7))
box(2.3, 11.4, 5.0, 1.3, "Sensitivity analysis: exclude extreme\nobservation (Tab Kwang); compare k-NN vs.\nfixed distance-band weights", color="#d9f0d9", edge="#2c6e2c")

arrow((4.8, 11.4), (4.8, 10.5))
box(2.3, 9.5, 5.0, 1.0, "Supplementary covariates & validation\n(2.3.3)", color="#fde6cf", edge="#a8611a")
box(0.3, 7.3, 4.3, 1.9, "Spatial covariates: distance to\nMittraphap Road & nearest cement\nplant, computed for every site\n(haversine)", color="#e6d9f0", edge="#5c2c8a")
box(5.0, 7.3, 4.7, 1.9, "Cross-validation: 32 field readings vs.\nsame-day PCD reference stations\n(24T, 25T)", color="#e6d9f0", edge="#5c2c8a")
arrow((4.8, 9.5), (2.45, 9.2)); arrow((4.8, 9.5), (7.35, 9.2))

arrow((2.45, 7.3), (4.8, 6.4)); arrow((7.35, 7.3), (4.8, 6.4))
stage_label(9.3, 5.6, "OUTPUT")
box(0.3, 4.3, 4.3, 1.3, "Descriptive stats & global\nautocorrelation (Table 1, 3)", color="#f7d9d9", edge="#8a2c2c")
box(4.9, 4.3, 4.8, 1.3, "Significant hotspots\n(Table 4, Fig. 5)", color="#f7d9d9", edge="#8a2c2c")
arrow((4.8, 6.4), (2.45, 5.6)); arrow((4.8, 6.4), (7.3, 5.6))

arrow((2.45, 4.3), (4.8, 3.4)); arrow((7.3, 4.3), (4.8, 3.4))
box(0.3, 2.1, 4.3, 1.3, "Sensitivity & meteorological\ncontext (Table 5\u20136)", color="#f7d9d9", edge="#8a2c2c")
box(4.9, 2.1, 4.8, 1.3, "Covariate & external\nvalidation results (Table 2, 7)", color="#f7d9d9", edge="#8a2c2c")

plt.tight_layout()
plt.savefig("figure2_flowchart.svg", format="svg", bbox_inches="tight")
plt.savefig("figure2_flowchart.png", format="png", dpi=220, bbox_inches="tight")
plt.show()
print("Saved figure2_flowchart.svg/.png")
print("NOTE: check the 'Table N' references baked into the OUTPUT boxes above still")
print("match the manuscript's current table numbering before using this figure --")
print("table numbers have been renumbered more than once during revisions.")


## 5. Figure 6 — Land-use context map


In [ ]:
# SCRIPT: Paper1_Figures_1_2_6_8.ipynb
# SECTION: 5 - Figure 6
fig, ax = plt.subplots(figsize=(8, 7.5))
study_amphoe.plot(ax=ax, color="#f7f7f7", edgecolor="#cccccc", linewidth=0.4)
study_province.boundary.plot(ax=ax, edgecolor="#555555", linewidth=1.0)
road_gdf.plot(ax=ax, color="#1a7a1a", linewidth=2.2, zorder=4, label="Mittraphap Road (Hwy 2)")

sc = ax.scatter(df["lon"], df["lat"], c=df["pm25"], s=95, cmap="YlOrRd",
                 edgecolor="black", linewidth=0.5, zorder=5,
                 norm=LogNorm(vmin=df["pm25"].min(), vmax=df["pm25"].max()))
ax.scatter(plants["lon"], plants["lat"], s=170, c="black", marker="s", zorder=7,
           edgecolor="white", linewidth=1.0, label="Verified cement plant")
for _, r in plants.iterrows():
    ax.annotate(r["name"], (r["lon"], r["lat"]), fontsize=7.5, xytext=(6, 6),
                textcoords="offset points", fontweight="bold")

cbar = plt.colorbar(sc, ax=ax, shrink=0.75, pad=0.02)
cbar.set_label("PM2.5 (\u00b5g/m\u00b3, log scale)", fontsize=9)

ax.set_xlim(lon_min, lon_max); ax.set_ylim(lat_min, lat_max)
ax.set_xlabel("Longitude (\u00b0E)"); ax.set_ylabel("Latitude (\u00b0N)")
ax.set_title("Land-use context: sampling sites, cement plants, and Mittraphap Road", fontsize=11)
ax.legend(loc="lower left", fontsize=8, framealpha=0.9)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig("figure6_landuse_context.svg", format="svg", bbox_inches="tight")
plt.savefig("figure6_landuse_context.png", format="png", dpi=220, bbox_inches="tight")
plt.show()
print("Saved figure6_landuse_context.svg/.png")


## 6. Figure 8 — PM2.5 vs. distance to nearest verified cement plant


In [ ]:
# SCRIPT: Paper1_Figures_1_2_6_8.ipynb
# SECTION: 6 - Figure 8
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

def dist_to_nearest_plant(lat, lon):
    return min(haversine(lat, lon, p.lat, p.lon) for p in plants.itertuples())

df["dist_plant_km"] = df.apply(lambda r: dist_to_nearest_plant(r["lat"], r["lon"]), axis=1)
df["log_pm25"] = np.log(df["pm25"])

r, p = pearsonr(df["log_pm25"], df["dist_plant_km"])
print(f"Pearson r = {r:.3f}, p = {p:.4f} (compare against Section 3.3 of the manuscript: r=-0.41, p=0.020)")

fig, ax = plt.subplots(figsize=(7, 5.5))
is_survey = df["is_survey"]
ax.scatter(df.loc[~is_survey, "dist_plant_km"], df.loc[~is_survey, "log_pm25"],
           s=60, c="#1f78b4", edgecolor="white", linewidth=0.5, label="Systematic-grid site", zorder=4)
ax.scatter(df.loc[is_survey, "dist_plant_km"], df.loc[is_survey, "log_pm25"],
           s=85, c="#e31a1c", marker="^", edgecolor="white", linewidth=0.5, label="Purposive risk site", zorder=5)

coef = np.polyfit(df["dist_plant_km"], df["log_pm25"], 1)
xline = np.linspace(df["dist_plant_km"].min(), df["dist_plant_km"].max(), 100)
ax.plot(xline, np.polyval(coef, xline), color="#333333", linestyle="--", linewidth=1.4, zorder=3,
        label=f"OLS fit (r = {r:.2f}, p = {p:.3f})")

tk = df.loc[df["pm25"].idxmax()]
ax.annotate("Tab Kwang\n(extreme value)", (tk["dist_plant_km"], tk["log_pm25"]),
            xytext=(10, -18), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="-", color="gray", lw=0.8))

ax.set_xlabel("Distance to nearest cement plant (km)")
ax.set_ylabel("log(PM2.5), \u00b5g/m\u00b3")
ax.set_title("PM2.5 vs. distance to nearest verified cement plant", fontsize=11)
ax.legend(loc="upper right", fontsize=8.5)

plt.tight_layout()
plt.savefig("figure8_pm25_vs_distance.svg", format="svg", bbox_inches="tight")
plt.savefig("figure8_pm25_vs_distance.png", format="png", dpi=220, bbox_inches="tight")
plt.show()
print("Saved figure8_pm25_vs_distance.svg/.png")


## ดาวน์โหลดผลลัพธ์


In [ ]:
# SCRIPT: Paper1_Figures_1_2_6_8.ipynb
# SECTION: Download output
from google.colab import files as gfiles
for f in ["figure1_study_area.svg", "figure1_study_area.png",
          "figure2_flowchart.svg", "figure2_flowchart.png",
          "figure6_landuse_context.svg", "figure6_landuse_context.png",
          "figure8_pm25_vs_distance.svg", "figure8_pm25_vs_distance.png"]:
    gfiles.download(f)
